In [1]:
#Create tmp folder to save numpy files

In [2]:
!mkdir -p tmp/convunet

In [3]:
!mkdir -p tmp/woong

# Write cody's code

In [4]:
%%writefile codygogo.py
import os
import random
from tqdm import tqdm
import pandas as pd
import numpy as np
from glob import glob
import gc
from collections import defaultdict
import time

import cv2
import torch
import torch.nn as nn
from torch.nn.parallel import DataParallel
from torch.utils.data import Dataset, DataLoader
from torch.optim import lr_scheduler
from torch.cuda import amp
import torch.optim as optim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed = 42):
    '''Sets the seed of the entire notebook so results are the same every time we run.
    This is for REPRODUCIBILITY.'''
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(42)

data_dir = \
    '/kaggle/input/blood-vessel-segmentation'

def file_to_id(f):
    s = f.split('/')
    return s[-3]+'_' + s[-1][:-4]

valid_meta = []
valid_folder = sorted(glob(f'{data_dir}/test/*'))
for image_folder in valid_folder:
    file = sorted(glob(f'{image_folder}/images/*.tif'))
    H, W = cv2.imread(file[0], cv2.IMREAD_ANYDEPTH).shape
    valid_meta.append(dict({
        'name':image_folder,
        'file':file,
        'shape':(len(file), H, W),
        'id':[file_to_id(f) for f in file],
    }))
        
        
print('len(valid_file) :', len(valid_meta))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from timm.models import create_model


class sSE(nn.Module):
    def __init__(self, out_channels):
        super(sSE, self).__init__()
        self.conv = nn.Sequential(nn.Conv2d(out_channels, 1, kernel_size=1,padding=0),
                                  nn.BatchNorm2d(1),
                                  nn.Sigmoid())
    def forward(self,x):
        x=self.conv(x)
        return x

class cSE(nn.Module):
    def __init__(self, out_channels):
        super(cSE, self).__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(out_channels, int(out_channels/2), kernel_size=1,padding=0),
                                   nn.BatchNorm2d(int(out_channels/2)),
                                   nn.ReLU(inplace=True),
                                   )
                        
        self.conv2 = nn.Sequential(nn.Conv2d(int(out_channels/2), out_channels, kernel_size=1,padding=0),
                                   nn.BatchNorm2d(out_channels),
                                   nn.Sigmoid(),
                                   )
    def forward(self,x):
        x=nn.AvgPool2d(x.size()[2:])(x)
        x=self.conv1(x)
        x=self.conv2(x)
        return x
    
class MyDecoderBlock(nn.Module):
    def __init__(
        self,
        in_channel,
        skip_channel,
        out_channel,
    ):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channel + skip_channel, out_channel, kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channel,out_channel,kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )
        self.spatial_gate = sSE(out_channel)
        self.channel_gate = cSE(out_channel)


    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        g1 = self.spatial_gate(x)
        g2 = self.channel_gate(x)
        x = g1*x + g2*x
        return x

class MyUnetDecoder(nn.Module):
    def __init__(self,
                 in_channel,
                 skip_channel,
                 out_channel,
                 ):
        super().__init__()

        self.center = nn.Identity()

        i_channel = [in_channel, ] + out_channel[:-1]
        s_channel = skip_channel
        o_channel = out_channel
        block = [
            MyDecoderBlock(i, s, o,)
            for i, s, o in zip(i_channel, s_channel, o_channel)
        ]
        self.block = nn.ModuleList(block)

    def forward(self, feature, skip):
        d = self.center(feature)

        for i, block in enumerate(self.block):
            s = skip[i]
            d = block(d, s)
            
        last = d
        return last


class ConvNeXt_U(nn.Module):
    def __init__(self):
        super().__init__() 
        encoder_dim = [24, 48, 96, 192, 384, 768]
        decoder_dim = [384, 192, 96, 48, 24]

        self.encoder = create_model('convnext_small.fb_in22k', pretrained=False, in_chans=1)

        self.decoder = MyUnetDecoder(
            in_channel  = encoder_dim[-1],
            skip_channel= encoder_dim[:-1][::-1],
            out_channel = decoder_dim,
        )
        self.vessel = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.kidney = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.stem0 = nn.Sequential(nn.Conv2d(in_channels=1, out_channels=24, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                   nn.Conv2d(in_channels=24, out_channels=24, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                  )
        self.stem1 = nn.Sequential(nn.Conv2d(in_channels=24, out_channels=48, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                   nn.Conv2d(in_channels=48, out_channels=48, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                  )

    def forward(self, image):
        B, C, H, W = image.shape
        H_pad = (32 - H % 32) % 32
        W_pad = (32 - W % 32) % 32
        x = F.pad(image, (0, W_pad, 0, H_pad), 'constant', 0)
        # x = x.expand(-1, 3, -1, -1)

        encode = []
        xx = self.stem0(x); encode.append(xx)
        xx = F.avg_pool2d(xx,kernel_size=2,stride=2)
        xx = self.stem1(xx); encode.append(xx)

        e = self.encoder
        x = e.stem(x);

        x = e.stages[0](x); encode.append(x)
        x = e.stages[1](x); encode.append(x)
        x = e.stages[2](x); encode.append(x)
        x = e.stages[3](x); encode.append(x)
        #[print(f'encode_{i}', e.shape) for i,e in enumerate(encode)]
        last = self.decoder(
            feature=encode[-1], skip=encode[:-1][::-1]
        )

        vessel = self.vessel(last).float()
        vessel = F.logsigmoid(vessel).exp()
        vessel = vessel[:, :, :H, :W].contiguous()
        
        kidney = self.kidney(last).float()
        kidney = F.logsigmoid(kidney).exp()
        kidney = kidney[:, :, :H, :W].contiguous()
        
        return vessel, kidney
    
def run_check_net():
    height, width = 260, 256
    batch_size = 2

    image = torch.from_numpy(np.random.uniform(0, 1, (batch_size, 1, height, width))).float().to(device)

    net = ConvNeXt_U().to(device)

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=True):
            v, k = net(image)

    print('image', image.shape)
    print('vessel', v.shape)
    print('kidney', k.shape)

model1 = ConvNeXt_U()
model1.load_state_dict(torch.load("/kaggle/input/sennet-hoa-models/convnext_small-unet-1ltuf51v.pt", map_location = device))
model1.eval()
    
model1 = model1.to(device)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from timm.models import create_model


class sSE(nn.Module):
    def __init__(self, out_channels):
        super(sSE, self).__init__()
        self.conv = nn.Sequential(nn.Conv2d(out_channels, 1, kernel_size=1,padding=0),
                                  nn.BatchNorm2d(1),
                                  nn.Sigmoid())
    def forward(self,x):
        x=self.conv(x)
        return x

class cSE(nn.Module):
    def __init__(self, out_channels):
        super(cSE, self).__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(out_channels, int(out_channels/2), kernel_size=1,padding=0),
                                   nn.BatchNorm2d(int(out_channels/2)),
                                   nn.ReLU(inplace=True),
                                   )
                        
        self.conv2 = nn.Sequential(nn.Conv2d(int(out_channels/2), out_channels, kernel_size=1,padding=0),
                                   nn.BatchNorm2d(out_channels),
                                   nn.Sigmoid(),
                                   )
    def forward(self,x):
        x=nn.AvgPool2d(x.size()[2:])(x)
        x=self.conv1(x)
        x=self.conv2(x)
        return x
    
class MyDecoderBlock(nn.Module):
    def __init__(
        self,
        in_channel,
        skip_channel,
        out_channel,
    ):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channel + skip_channel, out_channel, kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channel,out_channel,kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )
        self.spatial_gate = sSE(out_channel)
        self.channel_gate = cSE(out_channel)


    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        g1 = self.spatial_gate(x)
        g2 = self.channel_gate(x)
        x = g1*x + g2*x
        return x

class MyUnetDecoder(nn.Module):
    def __init__(self,
                 in_channel,
                 skip_channel,
                 out_channel,
                 ):
        super().__init__()

        self.center = nn.Identity()

        i_channel = [in_channel, ] + out_channel[:-1]
        s_channel = skip_channel
        o_channel = out_channel
        block = [
            MyDecoderBlock(i, s, o,)
            for i, s, o in zip(i_channel, s_channel, o_channel)
        ]
        self.block = nn.ModuleList(block)

    def forward(self, feature, skip):
        d = self.center(feature)

        for i, block in enumerate(self.block):
            s = skip[i]
            d = block(d, s)
            
        last = d
        return last


class ConvNeXt_U(nn.Module):
    def __init__(self):
        super().__init__() 
        encoder_dim = [24, 48, 96, 192, 384, 768]
        decoder_dim = [384, 192, 96, 48, 24]

        self.encoder = create_model('convnext_small.fb_in22k', pretrained=False, in_chans=3)

        self.decoder = MyUnetDecoder(
            in_channel  = encoder_dim[-1],
            skip_channel= encoder_dim[:-1][::-1],
            out_channel = decoder_dim,
        )
        self.vessel = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.kidney = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.stem0 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=24, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                   nn.Conv2d(in_channels=24, out_channels=24, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                  )
        self.stem1 = nn.Sequential(nn.Conv2d(in_channels=24, out_channels=48, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                   nn.Conv2d(in_channels=48, out_channels=48, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                  )

    def forward(self, image):
        B, C, H, W = image.shape
        H_pad = (32 - H % 32) % 32
        W_pad = (32 - W % 32) % 32
        x = F.pad(image, (0, W_pad, 0, H_pad), 'constant', 0)
        x = x.expand(-1, 3, -1, -1)

        encode = []
        xx = self.stem0(x); encode.append(xx)
        xx = F.avg_pool2d(xx,kernel_size=2,stride=2)
        xx = self.stem1(xx); encode.append(xx)

        e = self.encoder
        x = e.stem(x);

        x = e.stages[0](x); encode.append(x)
        x = e.stages[1](x); encode.append(x)
        x = e.stages[2](x); encode.append(x)
        x = e.stages[3](x); encode.append(x)
        #[print(f'encode_{i}', e.shape) for i,e in enumerate(encode)]
        last = self.decoder(
            feature=encode[-1], skip=encode[:-1][::-1]
        )

        vessel = self.vessel(last).float()
        vessel = F.logsigmoid(vessel).exp()
        vessel = vessel[:, :, :H, :W].contiguous()
        
        kidney = self.kidney(last).float()
        kidney = F.logsigmoid(kidney).exp()
        kidney = kidney[:, :, :H, :W].contiguous()
        
        return vessel, kidney
    
def run_check_net():
    height, width = 260, 256
    batch_size = 2

    image = torch.from_numpy(np.random.uniform(0, 1, (batch_size, 1, height, width))).float().to(device)

    net = ConvNeXt_U().to(device)

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=True):
            v, k = net(image)

    print('image', image.shape)
    print('vessel', v.shape)
    print('kidney', k.shape)


model2 = ConvNeXt_U()
model2.load_state_dict(torch.load("/kaggle/input/sennet-hoa-models/convnext_small-unet-1cbx0ckf.pt", map_location = device))
model2.eval()
model2 = model2.to(device)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from timm.models import create_model


class MyDecoderBlock(nn.Module):
    def __init__(
        self,
        in_channel,
        skip_channel,
        out_channel,
    ):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channel + skip_channel, out_channel, kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channel,out_channel,kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )


    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class MyUnetDecoder(nn.Module):
    def __init__(self,
                 in_channel,
                 skip_channel,
                 out_channel,
                 ):
        super().__init__()

        self.center = nn.Identity()

        i_channel = [in_channel, ] + out_channel[:-1]
        s_channel = skip_channel
        o_channel = out_channel
        block = [
            MyDecoderBlock(i, s, o,)
            for i, s, o in zip(i_channel, s_channel, o_channel)
        ]
        self.block = nn.ModuleList(block)

    def forward(self, feature, skip):
        d = self.center(feature)

        for i, block in enumerate(self.block):
            s = skip[i]
            d = block(d, s)
            
        last = d
        return last


class ConvNeXt_U(nn.Module):
    def __init__(self):
        super().__init__() 
        encoder_dim = [24, 48, 96, 192, 384, 768]
        decoder_dim = [384, 192, 96, 48, 24]

        self.encoder = create_model('convnext_small.fb_in22k', pretrained=False, in_chans=3)

        self.decoder = MyUnetDecoder(
            in_channel  = encoder_dim[-1],
            skip_channel= encoder_dim[:-1][::-1],
            out_channel = decoder_dim,
        )
        self.vessel = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.kidney = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.stem0 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=24, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), nn.ReLU(inplace=True))
        self.stem1 = nn.Sequential(nn.Conv2d(in_channels=24, out_channels=48, kernel_size=3, stride=1, padding=1), nn.BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), nn.ReLU(inplace=True))

    def forward(self, image):
        B, C, H, W = image.shape
        H_pad = (32 - H % 32) % 32
        W_pad = (32 - W % 32) % 32
        x = F.pad(image, (0, W_pad, 0, H_pad), 'constant', 0)
        x = x.expand(-1, 3, -1, -1)

        encode = []
        xx = self.stem0(x); encode.append(xx)
        xx = F.avg_pool2d(xx,kernel_size=2,stride=2)
        xx = self.stem1(xx); encode.append(xx)

        e = self.encoder
        x = e.stem(x);

        x = e.stages[0](x); encode.append(x)
        x = e.stages[1](x); encode.append(x)
        x = e.stages[2](x); encode.append(x)
        x = e.stages[3](x); encode.append(x)
        #[print(f'encode_{i}', e.shape) for i,e in enumerate(encode)]
        last = self.decoder(
            feature=encode[-1], skip=encode[:-1][::-1]
        )

        vessel = self.vessel(last).float()
        vessel = F.logsigmoid(vessel).exp()
        vessel = vessel[:, :, :H, :W].contiguous()
        
        kidney = self.kidney(last).float()
        kidney = F.logsigmoid(kidney).exp()
        kidney = kidney[:, :, :H, :W].contiguous()
        
        return vessel, kidney

model3 = ConvNeXt_U()
model3.load_state_dict(torch.load("/kaggle/input/sennet-hoa-models/convnext_small-unet-16ylc1b5.pt", map_location = device))
model3.eval()
model3 = model3.to(device)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from timm.models import create_model

class sSE(nn.Module):
    def __init__(self, out_channels):
        super(sSE, self).__init__()
        self.conv = nn.Sequential(nn.Conv2d(out_channels, 1, kernel_size=1,padding=0),
                                  nn.BatchNorm2d(1),
                                  nn.Sigmoid())
    def forward(self,x):
        x=self.conv(x)
        return x

class cSE(nn.Module):
    def __init__(self, out_channels):
        super(cSE, self).__init__()
        self.conv1 = nn.Sequential(nn.Conv2d(out_channels, int(out_channels/2), kernel_size=1,padding=0),
                                   nn.BatchNorm2d(int(out_channels/2)),
                                   nn.ReLU(inplace=True),
                                   )
                        
        self.conv2 = nn.Sequential(nn.Conv2d(int(out_channels/2), out_channels, kernel_size=1,padding=0),
                                   nn.BatchNorm2d(out_channels),
                                   nn.Sigmoid(),
                                   )
    def forward(self,x):
        x=nn.AvgPool2d(x.size()[2:])(x)
        x=self.conv1(x)
        x=self.conv2(x)
        return x
    
class MyDecoderBlock(nn.Module):
    def __init__(
        self,
        in_channel,
        skip_channel,
        out_channel,
    ):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channel + skip_channel, out_channel, kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channel,out_channel,kernel_size=3, padding=1,),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )
#         self.spatial_gate = sSE(out_channel)
#         self.channel_gate = cSE(out_channel)


    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear')
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
#         g1 = self.spatial_gate(x)
#         g2 = self.channel_gate(x)
#         x = g1*x + g2*x
        return x

class MyUnetDecoder(nn.Module):
    def __init__(self,
                 in_channel,
                 skip_channel,
                 out_channel,
                 ):
        super().__init__()

        self.center = nn.Identity()

        i_channel = [in_channel, ] + out_channel[:-1]
        s_channel = skip_channel
        o_channel = out_channel
        block = [
            MyDecoderBlock(i, s, o,)
            for i, s, o in zip(i_channel, s_channel, o_channel)
        ]
        self.block = nn.ModuleList(block)

    def forward(self, feature, skip):
        d = self.center(feature)

        for i, block in enumerate(self.block):
            s = skip[i]
            d = block(d, s)
            
        last = d
        return last
    
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from timm.models import create_model
class ConvNeXt_U(nn.Module):
    def __init__(self):
        super().__init__() 
        encoder_dim = [24, 48, 96, 192, 384, 768]
        decoder_dim = [384, 192, 96, 48, 24]

        self.encoder = create_model('convnext_small.fb_in22k', pretrained=False, in_chans=3)

        self.decoder = MyUnetDecoder(
            in_channel  = encoder_dim[-1],
            skip_channel= encoder_dim[:-1][::-1],
            out_channel = decoder_dim,
        )
        self.vessel = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.kidney = nn.Conv2d(decoder_dim[-1], 1, kernel_size=1)
        self.stem0 = nn.Sequential(nn.Conv2d(in_channels=3, out_channels=24, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                   )
        self.stem1 = nn.Sequential(nn.Conv2d(in_channels=24, out_channels=48, kernel_size=3, stride=1, padding=1), 
                                   nn.BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True), 
                                   nn.ReLU(inplace=True),
                                   )

    def forward(self, image):
        _, _, H, W = image.shape
        H_pad = (32 - H % 32) % 32
        W_pad = (32 - W % 32) % 32
        x = F.pad(image, (0, W_pad, 0, H_pad), 'constant', 0)
        x = x.expand(-1, 3, -1, -1)

        encode = []
        xx = self.stem0(x); encode.append(xx)
        xx = F.avg_pool2d(xx,kernel_size=2,stride=2)
        xx = self.stem1(xx); encode.append(xx)

        e = self.encoder
        x = e.stem(x);

        x = e.stages[0](x); encode.append(x)
        x = e.stages[1](x); encode.append(x)
        x = e.stages[2](x); encode.append(x)
        x = e.stages[3](x); encode.append(x)
        #[print(f'encode_{i}', e.shape) for i,e in enumerate(encode)]
        last = self.decoder(
            feature=encode[-1], skip=encode[:-1][::-1]
        )

        vessel = self.vessel(last).float()
        vessel = F.logsigmoid(vessel).exp()
        vessel = vessel[:, :, :H, :W].contiguous()
        
        kidney = self.kidney(last).float()
        kidney = F.logsigmoid(kidney).exp()
        kidney = kidney[:, :, :H, :W].contiguous()
        
        return vessel, kidney
    
def run_check_net():
    height, width = 260, 256
    batch_size = 2

    image = torch.from_numpy(np.random.uniform(0, 1, (batch_size, 1, height, width))).float().cuda()

    net = ConvNeXt_U().cuda()

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=True):
            v, k = net(image)

    print('image', image.shape)
    print('vessel', v.shape)
    print('kidney', k.shape)

model_cody = ConvNeXt_U()

state_dict = torch.load("/kaggle/input/cody-convnext/convnext-unet-6fr2vqem.pt")
model_dict = model_cody.state_dict()
filtered_state_dict = {k: v for k, v in state_dict.items() if k in model_dict}
model_cody.load_state_dict(filtered_state_dict)

model_cody.eval()
model_cody = model_cody.to(device)

def rle_encode(mask):
    pixel = mask.flatten()
    pixel = np.concatenate([[0], pixel, [0]])
    run = np.where(pixel[1:] != pixel[:-1])[0] + 1
    run[1::2] -= run[::2]
    rle = ' '.join(str(r) for r in run)
    if rle == '':
        rle = '1 0'
    return rle

def make_prediction(predict, model, volume, shape, axes):
    model = DataParallel(model)
    for axis in axes:
        loader = np.array_split(np.arange((D, H, W)[axis]), max(1, int((D, H, W)[axis] // 8)))
        num_valid = len(loader)

        B = 0 
        for t in range(num_valid):

            if axis == 0:
                image = volume[loader[t].tolist()]
            if axis == 1:
                image = volume[:, loader[t].tolist()]
                image = image.transpose(1, 0, 2)
            if axis == 2:
                image = volume[:, :, loader[t].tolist()]
                image = image.transpose(2, 0, 1)

            batch_size, bh, bw = image.shape
            
            images = np.zeros_like(image)

            for i in range(batch_size):
                img = image[i]
                img = img.reshape(-1)
                img = (img - img.min()) / (img.max() - img.min() + 0.0001)
                img = img.reshape(bh, bw)
                img = np.ascontiguousarray(img)
                images[i] = img
            image = torch.from_numpy(images).float().to(device).unsqueeze(1)

            counter = 0
            vessel1, kidney1 = 0, 0
            vessel2, kidney2 = 0, 0
            image = image.to(device)
            with torch.cuda.amp.autocast():
                with torch.no_grad():
                    v1, k1 = model(image)
                    vessel1 += v1
                    kidney1 += k1
                    counter += 1

                    v1, k1 = model(torch.flip(image, dims=[2,]))
                    vessel1 += torch.flip(v1, dims=[2,])
                    kidney1 += torch.flip(k1, dims=[2,])
                    counter += 1

                    v1, k1 = model(torch.flip(image, dims=[3,]))
                    vessel1 += torch.flip(v1, dims=[3,])
                    kidney1 += torch.flip(k1, dims=[3,])
                    counter += 1

                    flipped_image = torch.flip(image, dims=[2, 3])
                    v1, k1 = model(flipped_image)
                    vessel1 += torch.flip(v1, dims=[2, 3])
                    kidney1 += torch.flip(k1, dims=[2, 3])
                    counter += 1

            vessel1 = vessel1/counter   
            kidney1 = kidney1/counter         

            vessel1 = vessel1.float().data.cpu().numpy()
            kidney1 = kidney1.float().data.cpu().numpy()
            kidney1 = kidney1 > 0.5

            batch_size = len(vessel1)
            for b in range(batch_size):
                mk1 = kidney1[b, 0]
                mv1 = vessel1[b, 0]

                p1 = (mv1 * mk1)
                if axis == 0:
                    predict[B + b] += (p1/3)
                if axis == 1:
                    predict[:, B + b] += (p1/3)
                if axis == 2:
                    predict[:, :, B + b] += (p1/3)
            B += batch_size
        
    del model

start = time.time()
submission_df = []
for d in valid_meta:
    predict = np.zeros(d['shape'], dtype=np.float16)
    volume = [cv2.imread(f, cv2.IMREAD_ANYDEPTH).astype(np.float32) for f in d['file']]
    volume = np.stack(volume)
    D, H, W = volume.shape

    axes = [0,1,2]
    
    #make_prediction(predict, model1, volume, (D, H, W), axes)
    #torch.cuda.empty_cache()
    #gc.collect()
    
    make_prediction(predict, model2, volume, (D, H, W), axes)
    #torch.cuda.empty_cache()
    #gc.collect()
    
    make_prediction(predict, model_cody, volume, (D, H, W), axes)
    #torch.cuda.empty_cache()
    #gc.collect()
    
    make_prediction(predict, model3, volume, (D, H, W), axes)
    torch.cuda.empty_cache()
    gc.collect()
    for idx in range(predict.shape[0]):
        np.save(os.path.join("tmp/convunet",str(d['id'][idx])+".npy"),predict[idx,:])

        

Writing codygogo.py


In [5]:
!python codygogo.py

len(valid_file) : 2


# Woong Model

In [6]:
!python -m pip install --no-index --find-links=/kaggle/input/pip-download-for-segmentation-models-pytorch segmentation-models-pytorch

Looking in links: /kaggle/input/pip-download-for-segmentation-models-pytorch
Processing /kaggle/input/pip-download-for-segmentation-models-pytorch/segmentation_models_pytorch-0.3.3-py3-none-any.whl
Processing /kaggle/input/pip-download-for-segmentation-models-pytorch/pretrainedmodels-0.7.4.tar.gz (from segmentation-models-pytorch)
  Preparing metadata (setup.py) ... - \ done
Processing /kaggle/input/pip-download-for-segmentation-models-pytorch/efficientnet_pytorch-0.7.1.tar.gz (from segmentation-models-pytorch)
  Preparing metadata (setup.py) ... - done
Processing /kaggle/input/pip-download-for-segmentation-models-pytorch/timm-0.9.2-py3-none-any.whl (from segmentation-models-pytorch)
Processing /kaggle/input/pip-download-for-segmentation-models-pytorch/munch-4.0.0-py2.py3-none-any.whl (from pretrainedmodels==0.7.4->segmentation-models-pytorch)
  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16428 sha256=05d549059b3c1187812da9c42

In [7]:
%%writefile woong.py

import torch as tc 
import torch.nn as nn  
import numpy as np
from tqdm import tqdm
from torch.cuda.amp import autocast
import cv2
import os,sys
from glob import glob
import matplotlib.pyplot as plt
import pandas as pd

import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
from torch.nn.parallel import DataParallel
from dotenv import load_dotenv


model_path_i = 0 #in_chans_1__25     3 in_chans_1__20 2 
class CFG:
# ============== model CFG =============
    model_name = 'Unet'
    backbone = 'tu-tf_efficientnetv2_m.in21k_ft_in1k'

    in_chans = 1 #5 # 65
    #============== _ CFG =============
    image_size = 768
    input_size=1024
    tile_size = image_size
    stride = tile_size // 2
    drop_egde_pixel=32
    
    target_size = 1
    chopping_percentile=1e-3
    # ============== fold =============
    valid_id = 1
    batch=32
    th_percentile = 0.00145 # 0.0014 #0.00175 #0.0021
    model_path=[
               "/kaggle/input/sennet-checkpoints/LAST_Unet_tu-tf_efficientnetv2_m.in21k_ft_in1k_99_loss0.05_score0.89_val_loss0.06_val_score0.87_max.pth"]# 25 025 rot 512 center

class CustomModel(nn.Module):
    def __init__(self, CFG, weight=None):
        super().__init__()
        self.CFG = CFG
        self.model = smp.Unet(
            encoder_name=CFG.backbone, 
            encoder_weights=weight,
            in_channels=CFG.in_chans,
            classes=CFG.target_size,
            activation=None,
        )
        self.batch=CFG.batch

    def forward_(self, image):
        output = self.model(image)
        return output[:,0]
    
    def forward(self,x:tc.Tensor):
        #x.shape=(batch,c,h,w)
        x=x.to(tc.float16)
        x=norm_with_clip(x.reshape(-1,*x.shape[2:])).reshape(x.shape)
        
        if CFG.input_size!=CFG.image_size:
            x=nn.functional.interpolate(x,size=(CFG.input_size,CFG.input_size),mode='bilinear',align_corners=True)
        
        shape=x.shape
        x=[tc.rot90(x,k=i,dims=(-2,-1)) for i in range(4)]
        x=tc.cat(x,dim=0)
        with autocast(enabled=True):
            with tc.no_grad():
                x=[self.forward_(x[i*self.batch:(i+1)*self.batch]) for i in range(x.shape[0]//self.batch+1)]
                # batch=64,64...48
                x=tc.cat(x,dim=0)
        x=x.sigmoid()
        x=x.reshape(4,shape[0],*shape[2:])
        x=[tc.rot90(x[i],k=-i,dims=(-2,-1)) for i in range(4)]
        x=tc.stack(x,dim=0).mean(0)
        
        if CFG.input_size!=CFG.image_size:
            x=nn.functional.interpolate(x[None],size=(CFG.image_size,CFG.image_size),mode='bilinear',align_corners=True)[0]
        return x


def build_model(weight=None):
    load_dotenv()

    print('model_name', CFG.model_name)
    print('backbone', CFG.backbone)

    model = CustomModel(CFG, weight)

    return model.cuda()

def rle_encode(mask):
    pixel = mask.flatten()
    pixel = np.concatenate([[0], pixel, [0]])
    run = np.where(pixel[1:] != pixel[:-1])[0] + 1
    run[1::2] -= run[::2]
    rle = ' '.join(str(r) for r in run)
    if rle == '':
        rle = '1 0'
    return rle

def min_max_normalization(x:tc.Tensor)->tc.Tensor:
    """input.shape=(batch,f1,...)"""
    shape=x.shape
    if x.ndim>2:
        x=x.reshape(x.shape[0],-1)
    
    min_=x.min(dim=-1,keepdim=True)[0]
    max_=x.max(dim=-1,keepdim=True)[0]
    if min_.mean()==0 and max_.mean()==1:
        return x.reshape(shape)
    
    x=(x-min_)/(max_-min_+1e-9)
    return x.reshape(shape)

def norm_with_clip(x:tc.Tensor,smooth=1e-5):
    dim=list(range(1,x.ndim))
    mean=x.mean(dim=dim,keepdim=True)
    std=x.std(dim=dim,keepdim=True)
    x=(x-mean)/(std+smooth)
    x[x>5]=(x[x>5]-5)*1e-3 +5
    x[x<-3]=(x[x<-3]+3)*1e-3-3
    return x

class Data_loader(Dataset):
    def __init__(self,path,s="/images/"):
        self.paths=glob(path+f"{s}*.tif")
        self.paths.sort()
        self.bool=s=="/labels/"
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self,index):
        img=cv2.imread(self.paths[index],cv2.IMREAD_GRAYSCALE)
        img=tc.from_numpy(img)
        if self.bool:
            img=img.to(tc.bool)
        else:
            img=img.to(tc.uint8)
        return img

def load_data(path,s):
    data_loader=Data_loader(path,s)
    data_loader=DataLoader(data_loader, batch_size=16, num_workers=2)
    data=[]
    for x in tqdm(data_loader):
        data.append(x)
    x=tc.cat(data,dim=0)
    ########################################################################
    TH=x.reshape(-1).numpy()
    index = -int(len(TH) * CFG.chopping_percentile)
    TH:int = np.partition(TH, index)[index]
    x[x>TH]=int(TH)
    ########################################################################
    TH=x.reshape(-1).numpy()
    index = -int(len(TH) * CFG.chopping_percentile)
    TH:int = np.partition(TH, -index)[-index]
    x[x<TH]=int(TH)
    ########################################################################
    #x=(min_max_normalization(x.to(tc.float16))*255).to(tc.uint8)
    return x

class Pipeline_Dataset(Dataset):
    def __init__(self,x,path):
        self.img_paths  = glob(path+"/images/*")
        self.img_paths.sort()
        self.in_chan = CFG.in_chans
        z=tc.zeros(self.in_chan//2,*x.shape[1:],dtype=x.dtype)
        self.x=tc.cat((z,x,z),dim=0)
        
    def __len__(self):
        return self.x.shape[0]-self.in_chan+1
    
    def __getitem__(self, index):
        x  = self.x[index:index+self.in_chan]
        return x,index
    
    def get_mark(self,index):
        id=self.img_paths[index].split("/")[-3:]
        id.pop(1)
        id="_".join(id)
        return id[:-4]
    
    def get_marks(self):
        ids=[]
        for index in range(len(self)):
            ids.append(self.get_mark(index))
        return ids

def add_edge(x:tc.Tensor,edge:int):
    #x=(C,H,W)
    #output=(C,H+2*edge,W+2*edge)
    mean_=int(x.to(tc.float32).mean())
    x=tc.cat([x,tc.ones([x.shape[0],edge,x.shape[2]],dtype=x.dtype,device=x.device)*mean_],dim=1)
    x=tc.cat([x,tc.ones([x.shape[0],x.shape[1],edge],dtype=x.dtype,device=x.device)*mean_],dim=2)
    x=tc.cat([tc.ones([x.shape[0],edge,x.shape[2]],dtype=x.dtype,device=x.device)*mean_,x],dim=1)
    x=tc.cat([tc.ones([x.shape[0],x.shape[1],edge],dtype=x.dtype,device=x.device)*mean_,x],dim=2)
    return x

model=build_model()
model.load_state_dict(tc.load(CFG.model_path[ model_path_i ],"cpu"))
model.cuda()
model.eval()
model=DataParallel(model)

def get_output():
    outputs=[]
    #if debug:
    #    paths=["/kaggle/input/blood-vessel-segmentation/train/kidney_2"]
    #else:
    paths=glob("/kaggle/input/blood-vessel-segmentation/test/*")
    outputs=[[],[]]
    for path in paths:
        x=load_data(path,"/images/")
        labels=tc.zeros_like(x,dtype=tc.float16)
        mark=Pipeline_Dataset(x,path).get_marks()
        for axis in [0,1,2]:
            debug_count=0
            if axis==0:
                x_=x
                labels_=labels
            elif axis==1:
                x_=x.permute(1,2,0)
                labels_=labels.permute(1,2,0)
            elif axis==2:
                x_=x.permute(2,0,1)
                labels_=labels.permute(2,0,1)
            if x.shape[0]==3 and axis!=0:
                break
            dataset=Pipeline_Dataset(x_,path)
            dataloader=DataLoader(dataset,batch_size=1,shuffle=False,num_workers=2)
            shape=dataset.x.shape[-2:]
            x1_list = np.arange(0, shape[0]+CFG.tile_size-CFG.tile_size+1, CFG.stride)
            y1_list = np.arange(0, shape[1]+CFG.tile_size-CFG.tile_size+1, CFG.stride)
            for img,index in tqdm(dataloader):
                #img=(1,C,H,W)
                img=img.to("cuda:0")
                img=add_edge(img[0],CFG.tile_size//2)[None]

                mask_pred = tc.zeros_like(img[:,0],dtype=tc.float32,device=img.device)
                mask_count = tc.zeros_like(img[:,0],dtype=tc.float32,device=img.device)

                indexs=[]
                chip=[]
                for y1 in y1_list:
                    for x1 in x1_list:
                        x2 = x1 + CFG.tile_size
                        y2 = y1 + CFG.tile_size
                        indexs.append([x1+CFG.drop_egde_pixel,x2-CFG.drop_egde_pixel,
                                       y1+CFG.drop_egde_pixel,y2-CFG.drop_egde_pixel])
                        chip.append(img[...,x1:x2,y1:y2])
                #with torch.no_grad(), autocast(enabled=True):
                y_preds = model.forward(tc.cat(chip)).to(device=0)

                if CFG.drop_egde_pixel:
                    y_preds=y_preds[...,CFG.drop_egde_pixel:-CFG.drop_egde_pixel,
                                        CFG.drop_egde_pixel:-CFG.drop_egde_pixel]
                for i,(x1,x2,y1,y2) in enumerate(indexs):
                    mask_pred[...,x1:x2, y1:y2] += y_preds[i]
                    mask_count[...,x1:x2, y1:y2] += 1

                mask_pred /= mask_count

                #Rrecover
                mask_pred=mask_pred[...,CFG.tile_size//2:-CFG.tile_size//2,CFG.tile_size//2:-CFG.tile_size//2]
                
                labels_[index]+=(mask_pred[0]/3).to(tc.float16).cpu()
                print(labels.data_ptr() == labels_.data_ptr())
                
        outputs[0].append(labels.detach().cpu().numpy())
        outputs[1].append(mark)
    return outputs

output,ids=get_output()

for each_output_list, each_id_list in zip(output, ids):
    idx = 0
    for each_output_np, each_id in zip(each_output_list, each_id_list):
        print(each_id)
        np.save(os.path.join("tmp/woong",str(each_id)+".npy"),each_output_np[idx,:])
        idx+=1

Writing woong.py


In [8]:
!python woong.py

model_name Unet
backbone tu-tf_efficientnetv2_m.in21k_ft_in1k
100%|█████████████████████████████████████████████| 3/3 [00:04<00:00,  1.57s/it]
kidney_5_0000
kidney_5_0001
kidney_5_0002
kidney_6_0000
kidney_6_0001
kidney_6_0002


# Go Ensemble

In [9]:
import os
import random
from tqdm import tqdm
import pandas as pd
import numpy as np
import glob
import gc
from collections import defaultdict
import time

In [10]:
def rle_encode(mask):
    pixel = mask.flatten()
    pixel = np.concatenate([[0], pixel, [0]])
    run = np.where(pixel[1:] != pixel[:-1])[0] + 1
    run[1::2] -= run[::2]
    rle = ' '.join(str(r) for r in run)
    if rle == '':
        rle = '1 0'
    return rle

In [11]:
submission_df = []


In [12]:
th = 0.5

In [13]:
#np.max(cody_np)

In [14]:
cody_numpy_files = sorted(glob.glob("tmp/convunet/*npy"))
woong_numpy_files = sorted(glob.glob("tmp/woong/*npy"))
for cody_np_file, woong_np_file in zip(cody_numpy_files, woong_numpy_files):
    cody_np = np.load(cody_np_file)
    woong_np = np.load(woong_np_file)
    predict = cody_np*0.7 + woong_np*0.3
    predict = (predict > th).astype(np.uint8)
    rle = rle_encode(predict)
    #print(os.path.splitext(os.path.basename(cody_np_file))[0])
    submission_df.append(
            pd.DataFrame(data={
                'id'  : os.path.splitext(os.path.basename(cody_np_file))[0],
                'rle' : rle,
                
            },index=[0])
        )

In [15]:
submission = pd.concat(submission_df)

In [16]:

submission.to_csv('submission.csv', index=False)

In [17]:
submission.head()

,id,rle
0,kidney_5_0000,1 0
0,kidney_5_0001,1 0
0,kidney_5_0002,1 0
0,kidney_6_0000,1 0
0,kidney_6_0001,1 0


In [18]:
# predict = (predict > 0.68).astype(np.uint8)
    
#     rle = [rle_encode(p) for p in predict]
    
#     submission_df.append(
#             pd.DataFrame(data={
#                 'id'  : d['id'],
#                 'rle' : rle,
#             })
#         )
    
#     del volume, predict, rle
#     gc.collect()
#     torch.cuda.empty_cache()
# print(time.time() - start)

In [19]:
#cody model excluded: 0.869
#model 1 excluded:    0.872
#model 2 excluded:    0.865
#model 3 excluded:    0.871

In [20]:
#submission = pd.concat(submission_df)
#submission.to_csv('submission.csv', index=False)

In [21]:
#submission.head()